In [ ]:
from google.colab import files
uploaded = files.upload()

Saving tes.json to tes (2).json


In [ ]:

import json
import shutil
import os
import pandas as pd


# Karena diunggah menggunakan files.upload(), filenya langsung berada di root folder Colab
file_name = "tes.json"

# Load JSON
with open(file_name, "r", encoding="utf-8") as f:
    data = json.load(f)

samples = []

# Memastikan kita mengambil list data utama, baik formatnya langsung list atau dibungkus key tertentu
if isinstance(data, dict) and "data" in data:
    items = data["data"]
elif isinstance(data, list):
    items = data
else:
    items = next((v for v in data.values() if isinstance(v, list)), [])

# Looping ekstrak QA dataset sesuai format barumu
for item in items:
    question = item.get("question", "")
    context = item.get("context", "")

    # Ambil dictionary answers
    answers_dict = item.get("answers", {})

    # Ambil list di dalam key "text"
    text_list = answers_dict.get("text", [])

    # Ekstrak teks jawaban pertama jika list-nya tersedia
    if len(text_list) > 0:
        answer = text_list[0]
    else:
        answer = None

    samples.append({
        "question": question,
        "context": context,
        "answer": answer
    })

# Convert ke DataFrame
df = pd.DataFrame(samples)

print("\nHasil 5 data pertama:")
print(df.head())


Hasil 5 data pertama:
                                            question  \
0  Berapa rasio anak perempuan di Indonesia yang ...   
1  Berapa persentase tingkat mutilasi alat kelami...   
2     Anak laki-laki di sekolah rentan terhadap apa?   
3  Berapa anggaran pemerintah untuk perlindungan ...   
4  Berapa persen anak di bawah usia 18 tahun yang...   

                                             context                  answer  
0  Satu dari sembilan anak perempuan di Indonesia...      Satu dari sembilan  
1  Tingkat mutilasi alat kelamin perempuan di Ind...               52 persen  
2  Anak laki-laki di sekolah rentan terhadap sera...          serangan fisik  
3  Anggaran pemerintah untuk perlindungan anak ku...  kurang dari 0,1 persen  
4  Sekitar 17 persen anak di bawah usia 18 tahun ...               17 persen  


In [ ]:
# Pastikan library ter-install
# !pip install transformers>=4.48.0 datasets evaluate accelerate scikit-learn torch pandas numpy

import json
import torch
import gc
import time
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

print("Libraries imported successfully.")

# ==========================================
# 1. PERSIAPAN DATA (LOAD JSON CUSTOM)
# ==========================================
print("\n--- 1. Data Preparation ---")

def load_custom_qa_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if isinstance(data, dict) and "data" in data:
        items = data["data"]
    elif isinstance(data, list):
        items = data
    else:
        items = next((v for v in data.values() if isinstance(v, list)), [])

    contexts, questions, answers, ids = [], [], [], []

    for i, item in enumerate(items):
        context = item.get("context", "")
        question = item.get("question", "")

        answers_dict = item.get("answers", {})
        text_list = answers_dict.get("text", [])
        start_list = answers_dict.get("answer_start", [])

        if len(text_list) == 0 or len(start_list) == 0:
            answer_starts = []
            answer_texts = []
        else:
            answer_starts = [start_list[0]]
            answer_texts = [text_list[0]]

        contexts.append(context)
        questions.append(question)
        answers.append({'answer_start': answer_starts, 'text': answer_texts})
        ids.append(item.get("article_id", str(i)))

    return Dataset.from_dict({
        'id': ids,
        'context': contexts,
        'question': questions,
        'answers': answers
    })

try:
    file_name = 'tes.json'
    hf_dataset = load_custom_qa_data(file_name)

    # Split 80% Train, 20% Test
    dataset_dict = hf_dataset.train_test_split(test_size=0.2, seed=42)
    print(f"Dataset berhasil dimuat! Jumlah data Train: {len(dataset_dict['train'])}, Test: {len(dataset_dict['test'])}")
except FileNotFoundError:
    raise FileNotFoundError(f"FILE TIDAK DITEMUKAN: Pastikan kamu sudah mengupload '{file_name}' menggunakan files.upload()!")

# ==========================================
# 2. HYPERPARAMETER
# ==========================================
print("\n--- 2. Defining Hyperparameters ---")
epochs = 2
batch_size = 8
learning_rate = 2e-5
max_length = 384
doc_stride = 128


# ==========================================
# FUNGSI EVALUASI METRIK QA (TOKEN LEVEL)
# ==========================================
def compute_qa_metrics(eval_pred):
    start_logits, end_logits = eval_pred.predictions
    labels = eval_pred.label_ids

    # Handle jika label_ids dibungkus dalam tuple/list oleh Trainer
    if isinstance(labels, (tuple, list)):
        start_positions, end_positions = labels[0], labels[1]
    else:
        start_positions = labels[:, 0]
        end_positions = labels[:, 1]

    # Ambil indeks token dengan logit tertinggi sebagai prediksi
    pred_starts = np.argmax(start_logits, axis=-1)
    pred_ends = np.argmax(end_logits, axis=-1)

    # 1. Start Position Accuracy
    start_acc = np.mean(pred_starts == start_positions)

    # 2. End Position Accuracy
    end_acc = np.mean(pred_ends == end_positions)

    # 3. Overall Accuracy (Kombinasi Posisi Start & End harus benar keduanya)
    overall_acc = np.mean((pred_starts == start_positions) & (pred_ends == end_positions))

    # 4. Exact Match Token (Sama dengan overall position accuracy di tingkat token)
    em_token = overall_acc

    # 5. F1 Token (Menghitung overlap posisi token prediksi vs posisi token aktual)
    f1_scores = []
    for p_start, p_end, t_start, t_end in zip(pred_starts, pred_ends, start_positions, end_positions):
        # Proteksi jika indeks berbalik
        if p_start > p_end: p_start, p_end = p_end, p_start
        if t_start > t_end: t_start, t_end = t_end, t_start

        pred_span = set(range(p_start, p_end + 1))
        true_span = set(range(t_start, t_end + 1))

        # Penanganan Kasus Khusus: Jika pertanyaan tidak memiliki jawaban (menunjuk ke token CLS / Indeks 0)
        if (t_start == 0 and t_end == 0) or len(true_span) == 0:
            if (p_start == 0 and p_end == 0) or len(pred_span) == 0:
                f1_scores.append(1.0)
            else:
                f1_scores.append(0.0)
            continue

        if (p_start == 0 and p_end == 0) or len(pred_span) == 0:
            f1_scores.append(0.0)
            continue

        # Hitung irisan token span
        overlap = len(pred_span.intersection(true_span))
        if overlap == 0:
            f1_scores.append(0.0)
        else:
            precision = overlap / len(pred_span)
            recall = overlap / len(true_span)
            f1 = 2 * (precision * recall) / (precision + recall)
            f1_scores.append(f1)

    f1_token = np.mean(f1_scores)

    return {
        "start_accuracy": start_acc,
        "end_accuracy": end_acc,
        "overall_accuracy": overall_acc,
        "exact_match_token": em_token,
        "f1_token": f1_token
    }


# ==========================================
# 3. FUNGSI TRAINING & EVALUASI (+ HARDWARE USAGE)
# ==========================================
def train_and_evaluate_qa(model_name, tokenizer_name, dict_dataset, output_dir_prefix="qa_results"):
    print(f"\n--- Memulai eksperimen QA untuk {model_name} ---")
    start_time = time.time()

    # Bersihkan memori GPU dan reset pemantau Peak VRAM sebelum mulai
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    gc.collect()

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    def prepare_train_features(examples):
        examples["question"] = [q.lstrip() for q in examples["question"]]

        tokenized_examples = tokenizer(
            examples["question"],
            examples["context"],
            truncation="only_second",
            max_length=max_length,
            stride=doc_stride,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding="max_length",
        )

        sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
        offset_mapping = tokenized_examples.pop("offset_mapping")

        tokenized_examples["start_positions"] = []
        tokenized_examples["end_positions"] = []

        for i, offsets in enumerate(offset_mapping):
            input_ids = tokenized_examples["input_ids"][i]
            cls_index = input_ids.index(tokenizer.cls_token_id)
            sequence_ids = tokenized_examples.sequence_ids(i)
            sample_index = sample_mapping[i]
            answers = examples["answers"][sample_index]

            if len(answers["answer_start"]) == 0:
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                start_char = answers["answer_start"][0]
                end_char = start_char + len(answers["text"][0])

                token_start_index = 0
                while sequence_ids[token_start_index] != 1:
                    token_start_index += 1
                token_end_index = len(input_ids) - 1
                while sequence_ids[token_end_index] != 1:
                    token_end_index -= 1

                if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                    tokenized_examples["start_positions"].append(cls_index)
                    tokenized_examples["end_positions"].append(cls_index)
                else:
                    while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                        token_start_index += 1
                    tokenized_examples["start_positions"].append(token_start_index - 1)

                    while offsets[token_end_index][1] >= end_char:
                        token_end_index -= 1
                    tokenized_examples["end_positions"].append(token_end_index + 1)

        return tokenized_examples

    print("Tokenisasi dataset train...")
    train_dataset = dict_dataset["train"].map(
        prepare_train_features, batched=True, remove_columns=dict_dataset["train"].column_names
    )

    print("Tokenisasi dataset test...")
    eval_dataset = dict_dataset["test"].map(
        prepare_train_features, batched=True, remove_columns=dict_dataset["test"].column_names
    )

    model = AutoModelForQuestionAnswering.from_pretrained(tokenizer_name)

    training_args = TrainingArguments(
        output_dir=f"./{output_dir_prefix}/{model_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        load_best_model_at_end=True,
        metric_for_best_model="f1_token",
        greater_is_better=True,
        report_to="none",
        use_cpu=not torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_qa_metrics,
        processing_class=tokenizer,
        data_collator=DefaultDataCollator(),
    )

    print(f"Training {model_name}...")
    train_result = trainer.train()

    print(f"Evaluating Metrics {model_name}...")
    eval_results = trainer.evaluate()

    print(f"Evaluating Metrics {model_name}...")

    # 1. Catat waktu mulai inferensi secara manual
    inf_start_time = time.time()
    eval_results = trainer.evaluate()
    manual_eval_runtime = time.time() - inf_start_time

    # --- PENGUMPULAN METRIK HARDWARE & RUNTIME ---
    total_time_sec = time.time() - start_time

    # 1. Train Runtime & Samples/sec
    train_runtime = train_result.metrics.get("train_runtime", 0.0)
    train_samples_per_sec = train_result.metrics.get("train_samples_per_second", 0.0)

    # 2. Inference Runtime & Samples/sec (Coba ambil dari HF, jika 0.0 gunakan penghitungan manual)
    eval_runtime = eval_results.get("eval_runtime", eval_results.get("runtime", manual_eval_runtime))
    if eval_runtime == 0.0:
        eval_runtime = manual_eval_runtime

    eval_samples_per_sec = eval_results.get("eval_samples_per_second", eval_results.get("samples_per_second", 0.0))
    if eval_samples_per_sec == 0.0 and eval_runtime > 0:
        # Hitung manual: Jumlah sampel test dibagi waktu eksekusi
        eval_samples_per_sec = len(eval_dataset) / eval_runtime

    # 3. Peak VRAM (dalam Megabytes)
    if torch.cuda.is_available():
        peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        peak_vram_mb = 0.0  # Jika jalan di CPU

    print(f"Selesai! Peak VRAM: {peak_vram_mb:.2f} MB | Train Runtime: {train_runtime:.2f}s | Eval Runtime: {eval_runtime:.2f}s")

    del model, trainer, tokenizer, train_dataset, eval_dataset
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()

    # Kembalikan semua hasil metrik untuk rekap dataframe
    return {
        "model_name": model_name,
        "eval_loss": eval_results["eval_loss"],
        "exact_match_token": eval_results["eval_exact_match_token"],
        "f1_token": eval_results["eval_f1_token"],
        "train_runtime_sec": round(train_runtime, 2),
        "train_samples_per_sec": round(train_samples_per_sec, 2),
        "inference_runtime_sec": round(eval_runtime, 2),
        "inference_samples_per_sec": round(eval_samples_per_sec, 2),
        "peak_vram_mb": round(peak_vram_mb, 2),
        "total_experiment_time_sec": round(total_time_sec, 2)
    }

# ==========================================
# 4. SIKLUS EKSEKUSI UTAMA
# ==========================================
print("\n--- 4. Menjalankan Komparasi QA ---")
experiment_results = []

bert_results = train_and_evaluate_qa(
    model_name="BERT Klasik", tokenizer_name="bert-base-uncased", dict_dataset=dataset_dict
)
experiment_results.append(bert_results)

modern_bert_results = train_and_evaluate_qa(
    model_name="ModernBERT", tokenizer_name="answerdotai/ModernBERT-base", dict_dataset=dataset_dict
)
experiment_results.append(modern_bert_results)

# ==========================================
# 5. REKAPITULASI HASIL
# ==========================================
results_df = pd.DataFrame(experiment_results)
print("\nExperiment Summary (Question Answering) dengan Metrik Performa & Hardware:")
print(results_df.to_string(index=False))

results_df.to_csv("qa_comparison_results.csv", index=False)
print("\nEksperimen QA Selesai! Hasil disimpan di 'qa_comparison_results.csv'")

Libraries imported successfully.

--- 1. Data Preparation ---
Dataset berhasil dimuat! Jumlah data Train: 637, Test: 160

--- 2. Defining Hyperparameters ---

--- 4. Menjalankan Komparasi QA ---

--- Memulai eksperimen QA untuk BERT Klasik ---
Tokenisasi dataset train...


Map:   0%|          | 0/637 [00:00<?, ? examples/s]

Tokenisasi dataset test...


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

... (20 outputs left out)

Training Loss,Validation Loss,Epoch,Start Accuracy,End Accuracy,Overall Accuracy,Exact Match Token,F1 Token
No log,3.137240,2,0.193750,0.237500,0.075000,0.075000,0.396024


Evaluating Metrics ModernBERT...


Training Loss,Validation Loss,Epoch,Start Accuracy,End Accuracy,Overall Accuracy,Exact Match Token,F1 Token
No log,3.137240,2,0.193750,0.237500,0.075000,0.075000,0.396024


Selesai! Peak VRAM: 5158.25 MB | Train Runtime: 286.53s | Eval Runtime: 5.32s

Experiment Summary (Question Answering) dengan Metrik Performa & Hardware:
 model_name  eval_loss  exact_match_token  f1_token  train_runtime_sec  train_samples_per_sec  inference_runtime_sec  inference_samples_per_sec  peak_vram_mb  total_experiment_time_sec
BERT Klasik   3.061422            0.06875  0.446516             199.76                   6.38                   3.95                      40.52       3187.62                      211.6
 ModernBERT   3.137240            0.07500  0.396024             286.53                   4.45                   5.32                      30.07       5158.25                      302.0

Eksperimen QA Selesai! Hasil disimpan di 'qa_comparison_results.csv'


In [ ]:
tokenizer_for_length = AutoTokenizer.from_pretrained("bert-base-uncased")

def get_token_length(example):
    return {"context_token_length": len(tokenizer_for_length(example["context"])["input_ids"])}

# Apply the function to the dataset
dataset_with_lengths = hf_dataset.map(get_token_length, num_proc=1) # num_proc=1 to avoid issues if not running in a multiprocessing environment

# Calculate the average token length
average_context_token_length = np.mean(dataset_with_lengths["context_token_length"])

print(f"Rata-rata panjang token pada setiap context: {average_context_token_length:.2f}")

Map:   0%|          | 0/797 [00:00<?, ? examples/s]

Rata-rata panjang token pada setiap context: 52.13


In [ ]:
print(results_df)

    model_name  eval_loss  exact_match_token  f1_token  train_runtime_sec  \
0  BERT Klasik   3.061422            0.06875  0.446516             199.76   
1   ModernBERT   3.137240            0.07500  0.396024             286.53   

   train_samples_per_sec  inference_runtime_sec  inference_samples_per_sec  \
0                   6.38                   3.95                      40.52   
1                   4.45                   5.32                      30.07   

   peak_vram_mb  total_experiment_time_sec  
0       3187.62                      211.6  
1       5158.25                      302.0  
